# E-Means: Architecture & DynamicsIn-depth inspection of the Meta FAIR E-Means algorithm:- Mini-batch streaming with automatic scaling-law parameter selection- Running sufficient statistics: cluster count mass $N_j$ and sum vector $S_j$- Dynamic half-cosine decay schedule- Dead cluster revival via proportional donor splitting- Exact 1-pass finalization

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt

from src.algorithms.emeans import EMeans
from src.datasets.synthetic import generate_gaussian_blobs
from src.datasets.preprocessing import preprocess_features

In [ ]:
X_raw, _, _ = generate_gaussian_blobs(n_samples=2000, n_features=2, n_clusters=4, cluster_std=1.2, random_state=42)
X_norm, _ = preprocess_features(X_raw)

em = EMeans(n_clusters=4, n_epochs=5, random_state=42)
em.fit(X_norm)

print("E-Means Execution Recipe:")
for param, val in em.recipe.items():
    print(f"  {param:22s}: {val}")

print(f"\nEpochs Executed:       {em.n_iter_}")
print(f"Final Inertia:         {em.inertia_:.2f}")
print(f"Runtime:               {em.total_time_:.4f}s")
print(f"Dead Clusters Revived: {em.dead_revivals_count_}")

In [ ]:
# Centroid positioning after 5 epochs
plt.figure(figsize=(6.5, 4.5), dpi=150)
plt.scatter(X_norm[:, 0], X_norm[:, 1], c=em.labels_, cmap="tab10", s=20, alpha=0.5, edgecolors="none")
plt.scatter(em.centroids_[:, 0], em.centroids_[:, 1], c="red", marker="X", s=150, edgecolor="black", linewidth=1.5, label="E-Means Centroids")
plt.title(f"E-Means Partition (T = {em.n_epochs} Epochs)", fontsize=11, fontweight="bold")
plt.xlabel("Feature 1 (Standardized)", fontsize=9)
plt.ylabel("Feature 2 (Standardized)", fontsize=9)
plt.legend(frameon=True)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Epoch-level progression
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), dpi=150)

ax1.plot(em.history_["iteration"], em.history_["inertia"], marker="o", color="#1f77b4", linewidth=2.0)
ax1.set_title("Inertia Trajectory vs Epoch", fontsize=10.5, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=8.5)
ax1.set_ylabel("Inertia", fontsize=8.5)
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(em.history_["iteration"], em.history_["alpha"], marker="s", color="#d95f02", linewidth=2.0)
ax2.set_title("Learning Rate alpha vs Epoch", fontsize=10.5, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=8.5)
ax2.set_ylabel("Step Size (alpha)", fontsize=8.5)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()